<a href="https://colab.research.google.com/github/Annaa74/Google-colab-models/blob/main/Modular_Python_Code_for_MLOps_Workflow_Stages.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import joblib
import os
import json
from flask import Flask, request, jsonify # For model deployment simulation

# Define file paths for artifacts
DATA_PATH = 'data/wine_quality.csv'
PROCESSED_DATA_PATH = 'data/processed_wine_data.csv'
MODEL_PATH = 'models/logistic_regression_model.pkl'
METRICS_PATH = 'reports/metrics.json'
REPORT_PATH = 'reports/classification_report.txt'

# --- 1. Data Ingestion & Preprocessing ---
def ingest_and_preprocess_data():
    """
    Simulates data ingestion and preprocessing.
    Loads a dataset, handles a simple preprocess (scaling), and splits data.
    In a real CI/CD pipeline, this step would likely be triggered by new data arrival.
    """
    print("--- Starting Data Ingestion & Preprocessing ---")
    try:
        # Load a sample dataset (e.g., Wine Quality from UCI)
        # For simplicity, we'll create a dummy CSV or use a built-in dataset
        # In a real scenario, this would fetch from a database or data lake
        from sklearn.datasets import load_wine
        wine = load_wine()
        df = pd.DataFrame(data=wine.data, columns=wine.feature_names)
        df['target'] = wine.target

        # Separate features (X) and target (y)
        X = df.drop('target', axis=1)
        y = df['target']

        # Split data into training and test sets
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y
        )

        # Scale features
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        # Convert back to DataFrame for consistency (optional, but good for inspection)
        X_train_processed = pd.DataFrame(X_train_scaled, columns=X.columns)
        X_test_processed = pd.DataFrame(X_test_scaled, columns=X.columns)

        # Save processed data (optional, but good for traceability in CI/CD)
        os.makedirs(os.path.dirname(PROCESSED_DATA_PATH), exist_ok=True)
        X_train_processed.to_csv(PROCESSED_DATA_PATH.replace('.csv', '_train.csv'), index=False)
        X_test_processed.to_csv(PROCESSED_DATA_PATH.replace('.csv', '_test.csv'), index=False)
        y_train.to_csv(PROCESSED_DATA_PATH.replace('.csv', '_ytrain.csv'), index=False)
        y_test.to_csv(PROCESSED_DATA_PATH.replace('.csv', '_ytest.csv'), index=False)

        print(f"Data ingested and processed successfully. Processed data saved to {os.path.dirname(PROCESSED_DATA_PATH)}")
        return X_train_processed, X_test_processed, y_train, y_test, scaler

    except Exception as e:
        print(f"Error during data ingestion/preprocessing: {e}")
        return None, None, None, None, None

# --- 2. Model Training ---
def train_model(X_train, y_train):
    """
    Trains a Logistic Regression model.
    In a real CI/CD pipeline, this would be a separate build step after data prep.
    """
    print("\n--- Starting Model Training ---")
    if X_train is None or y_train is None:
        print("Training data not available. Skipping model training.")
        return None

    try:
        model = LogisticRegression(max_iter=1000, random_state=42)
        model.fit(X_train, y_train)

        # Save the trained model
        os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)
        joblib.dump(model, MODEL_PATH)
        print(f"Model trained and saved to {MODEL_PATH}")
        return model

    except Exception as e:
        print(f"Error during model training: {e}")
        return None

# --- 3. Model Evaluation ---
def evaluate_model(model, X_test, y_test):
    """
    Evaluates the trained model and saves metrics/report.
    In a real CI/CD pipeline, this step runs after training.
    """
    print("\n--- Starting Model Evaluation ---")
    if model is None or X_test is None or y_test is None:
        print("Model or test data not available. Skipping model evaluation.")
        return

    try:
        y_pred = model.predict(X_test)

        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
        recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
        f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)

        metrics = {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1_score': f1
        }

        # Save metrics
        os.makedirs(os.path.dirname(METRICS_PATH), exist_ok=True)
        with open(METRICS_PATH, 'w') as f:
            json.dump(metrics, f, indent=4)
        print(f"Evaluation metrics saved to {METRICS_PATH}")

        # Save classification report
        report = classification_report(y_test, y_pred, zero_division=0)
        os.makedirs(os.path.dirname(REPORT_PATH), exist_ok=True)
        with open(REPORT_PATH, 'w') as f:
            f.write(report)
        print(f"Classification report saved to {REPORT_PATH}")
        print("\n--- Model Evaluation Results ---")
        print(f"Accuracy: {accuracy:.4f}")
        print(f"Precision: {precision:.4f}")
        print(f"Recall: {recall:.4f}")
        print(f"F1 Score: {f1:.4f}")

        # In a CI/CD system, these metrics would be compared against a baseline
        # to decide if the new model is good enough for deployment.
        # Threshold checks would happen here.
        if accuracy > 0.90: # Example threshold
            print("\nModel performance meets deployment criteria!")
            return True
        else:
            print("\nModel performance does NOT meet deployment criteria.")
            return False

    except Exception as e:
        print(f"Error during model evaluation: {e}")
        return False

# --- 4. Model Versioning (Simulated) ---
# In a real MLOps setup, a tool like MLflow or DVC would handle this.
# For this example, saving the model and metrics to unique paths
# (e.g., including a timestamp or version number) would simulate versioning.
# E.g., MODEL_PATH = f'models/v_1.0/logistic_regression_model.pkl'

# --- 5. Model Deployment (Simulated API) ---
# This part would typically be a separate microservice.
# We'll include a simple Flask app structure to demonstrate.
app = Flask(__name__)
loaded_model = None
loaded_scaler = None

def load_model_for_serving():
    """
    Loads the trained model and scaler into memory for serving predictions.
    In a real deployment, this would happen when the service starts.
    """
    global loaded_model, loaded_scaler
    if loaded_model is None:
        try:
            loaded_model = joblib.load(MODEL_PATH)
            # Assuming the scaler was trained on the same data and needed for inference
            # For simplicity, we'll reuse the scaler from the data ingestion step if available.
            # In a real pipeline, the scaler would also be versioned and loaded.
            print(f"Model loaded successfully for serving from {MODEL_PATH}")
        except Exception as e:
            print(f"Error loading model for serving: {e}")
            loaded_model = None
    if loaded_scaler is None:
        # For this demo, let's assume the scaler instance is passed or re-instantiated
        # In a full pipeline, scaler state should also be saved and loaded with the model.
        # For now, we'll rely on the scaler from the main execution flow for simplicity.
        # A more robust solution would save/load the scaler state (e.g., joblib.dump(scaler, 'models/scaler.pkl'))
        pass # Placeholder for scaler loading logic

@app.route('/predict', methods=['POST'])
def predict():
    """
    API endpoint for making predictions.
    Expects a JSON payload with features.
    """
    if loaded_model is None:
        return jsonify({"error": "Model not loaded. Please run the workflow first."}), 500

    try:
        data = request.get_json(force=True)
        # Expecting input as a list of dictionaries, or single dictionary
        # Example: {"features": [5.1, 3.5, 1.4, 0.2, ...]}
        # Or {"features": [[...], [...]]} for batch
        input_data = pd.DataFrame(data['features']) # Ensure correct column names if needed

        # Apply scaling if a scaler was used during training
        if loaded_scaler:
            input_data_scaled = loaded_scaler.transform(input_data)
        else:
            input_data_scaled = input_data # If no scaler was used, or not loaded

        prediction = loaded_model.predict(input_data_scaled)
        # For multi-class classification, you might want probabilities too
        # probabilities = loaded_model.predict_proba(input_data_scaled).tolist()

        return jsonify({
            "prediction": prediction.tolist()
            # "probabilities": probabilities
        })

    except Exception as e:
        return jsonify({"error": str(e), "message": "Invalid input or prediction error."}), 400

# --- Main Workflow Execution (Simulating CI/CD flow) ---
if __name__ == "__main__":
    print("--- Starting MLOps Workflow Simulation ---")

    # Step 1: Data Ingestion & Preprocessing
    X_train, X_test, y_train, y_test, data_scaler = ingest_and_preprocess_data()
    # In a CI pipeline, if data ingestion fails, the pipeline would stop.
    if X_train is None:
        print("Data ingestion failed. Exiting workflow.")
    else:
        # Store scaler for later use in prediction
        loaded_scaler = data_scaler

        # Step 2: Model Training
        model = train_model(X_train, y_train)
        # In a CI pipeline, if training fails, the pipeline would stop.
        if model is None:
            print("Model training failed. Exiting workflow.")
        else:
            # Step 3: Model Evaluation
            # This step would output metrics, and based on thresholds,
            # decide if the model is fit for deployment (CD trigger).
            model_passed_evaluation = evaluate_model(model, X_test, y_test)

            if model_passed_evaluation:
                print("\n--- Model ready for deployment ---")
                # Step 4 (Conceptual): Model Versioning & Registry
                # A CI/CD tool (e.g., Jenkins, GitHub Actions) would now trigger
                # a step to register this model version and potentially promote it.
                # Example:
                # `dvc push` (for data/model versioning)
                # `mlflow.register_model()`

                # Step 5 (Conceptual): Model Deployment
                # In a real CD pipeline, this would trigger the deployment of the API service.
                # Here, we'll demonstrate by loading the model and starting the Flask app.
                print("Loading model for API serving...")
                load_model_for_serving()
                if loaded_model:
                    print("Model serving API is ready. Run the Flask app.")
                    # To run the Flask app:
                    # from werkzeug.serving import run_simple
                    # run_simple('127.0.0.1', 5000, app)
                    # For demonstration in Colab, you might use ngrok or similar.
                    # Or simply explain how to run it externally.
                    print("To test the API, save this code as `app.py` and run: `python -m flask run`")
                    print("Then send a POST request to http://127.0.0.1:5000/predict with JSON payload like:")
                    print('{"features": [[5.1, 3.5, 1.4, 0.2, 0.1, 1.5, 0.0, 1.0, 0.5, 2.0, 0.1, 1.0, 3.0]]}')
                    print("(Adjust features to match your dataset's columns/length)")

                else:
                    print("Failed to load model for serving. Deployment aborted.")
            else:
                print("Model failed evaluation. Skipping deployment.")

    print("\n--- MLOps Workflow Simulation Completed ---")